# Projeto de Ingestão — Bronze, Silver e Gold

Curso Databricks: Engenharia de Machine Learning e MLOps.

## Antes de executar no Databricks

1. Confirme que o catálogo `workspace` está disponível.
2. Crie o esquema `bronze`, se necessário.
3. Crie o Volume `workspace.bronze.raw`.
4. Envie os dois CSVs da pasta local `dados/upload_bronze`.
5. Importe este notebook e execute as células na ordem.

As tabelas existentes não serão sobrescritas.

In [ ]:
import re

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("volume", "raw")

catalogo = dbutils.widgets.get("catalogo").strip()
volume = dbutils.widgets.get("volume").strip()

for valor in (catalogo, volume):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError("Identificador invalido")

origem = f"/Volumes/{catalogo}/bronze/{volume}"

t_eventos = f"{catalogo}.bronze.tabela_eventos_bronze"
t_produtos = f"{catalogo}.bronze.dim_produtos"
t_silver = f"{catalogo}.silver.tabela_eventos_silver"
t_gold = f"{catalogo}.gold.base_eventos_produto_gold"

print("CATALOGO:", catalogo)
print("VOLUME:", origem)
print("BRONZE:", t_eventos, t_produtos)
print("SILVER:", t_silver)
print("GOLD:", t_gold)

## Etapa 1 — Verificação dos arquivos no Volume

O notebook interrompe a execução caso os CSVs ainda não tenham
sido enviados ao Databricks.

In [ ]:
esperados = {
    "base_bronze_eventos.csv",
    "base_dim_produtos.csv"
}

try:
    encontrados = {
        item.name.rstrip("/")
        for item in dbutils.fs.ls(origem)
    }
except Exception as erro:
    raise RuntimeError(
        f"Volume indisponivel: {origem}. "
        "Crie o Volume e envie os CSVs."
    ) from erro

faltando = esperados - encontrados

if faltando:
    raise FileNotFoundError(
        f"Arquivos ausentes no Volume: {sorted(faltando)}"
    )

print("VOLUME VALIDADO: 2/2 ARQUIVOS")

## Etapa 2 — Leitura dos CSVs com esquema explícito

A base de eventos utiliza ponto e vírgula.

A dimensão de produtos utiliza vírgula.

O Spark será configurado para interpretar as aspas escapadas
do JSON armazenado no CSV.

In [ ]:
schema_eventos = StructType([
    StructField("id_evento", StringType(), True),
    StructField("id_usuario", StringType(), True),
    StructField("tipo_evento", StringType(), True),
    StructField("timestamp_registro", StringType(), True),
    StructField("dados_origem_json", StringType(), True)
])

schema_produtos = StructType([
    StructField("id_produto", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("preco_reais", DecimalType(10, 2), True)
])

df_eventos = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .option("quote", '"')
    .option("escape", '"')
    .schema(schema_eventos)
    .csv(origem + "/base_bronze_eventos.csv")
)

df_produtos = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("mode", "FAILFAST")
    .schema(schema_produtos)
    .csv(origem + "/base_dim_produtos.csv")
)

n_eventos = df_eventos.count()
n_produtos = df_produtos.count()

if (n_eventos, n_produtos) != (20000, 15):
    raise ValueError(
        f"Contagens incorretas: {n_eventos}, {n_produtos}"
    )

if df_eventos.select("id_evento").distinct().count() != n_eventos:
    raise ValueError("Eventos duplicados")

if df_produtos.select("id_produto").distinct().count() != n_produtos:
    raise ValueError("Produtos duplicados")

if df_produtos.filter(
    F.col("preco_reais").isNull()
).limit(1).count():
    raise ValueError("Preco invalido")

id_json = F.get_json_object(
    F.col("dados_origem_json"), "$.id_produto"
)

if df_eventos.filter(id_json.isNull()).limit(1).count():
    raise ValueError("JSON invalido ou sem id_produto")

print("EVENTOS VALIDADOS:", n_eventos)
print("PRODUTOS VALIDADOS:", n_produtos)

## Etapa 3 — Criação das camadas e tabelas Bronze

A criação preserva tabelas existentes.

Se uma tabela preexistente tiver contagem ou colunas diferentes,
o notebook interromperá a execução para evitar sobrescritas.

In [ ]:
for esquema in ("bronze", "silver", "gold"):
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}"
    )


def salvar_sem_sobrescrever(df, nome, esperado):

    if spark.catalog.tableExists(nome):

        existente = spark.table(nome)
        quantidade = existente.count()

        if quantidade != esperado:
            raise RuntimeError(
                f"Tabela existente divergente: {nome}"
            )

        if set(existente.columns) != set(df.columns):
            raise RuntimeError(
                f"Colunas diferentes na tabela: {nome}"
            )

        print("PRESERVADA:", nome, quantidade)
        return existente

    (
        df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(nome)
    )

    print("CRIADA:", nome, esperado)
    return spark.table(nome)


bronze_eventos = salvar_sem_sobrescrever(
    df_eventos, t_eventos, n_eventos
)

bronze_produtos = salvar_sem_sobrescrever(
    df_produtos, t_produtos, n_produtos
)

print("CAMADA BRONZE CONFERIDA")

## Etapa 4 — Silver: tratamento e estruturação do JSON

O timestamp será convertido.

Os campos JSON serão transformados em colunas.

Não será realizada limpeza indiscriminada das aspas,
pois o leitor CSV já interpreta o escape do arquivo.

In [ ]:
schema_json = StructType([
    StructField("id_produto", StringType(), True),
    StructField("dispositivo", StringType(), True),
    StructField("sistema_operacional", StringType(), True),
    StructField("origem_trafego", StringType(), True),
    StructField("idade", IntegerType(), True),
    StructField("genero", StringType(), True),
    StructField("renda_mensal_estimada_k", DecimalType(18, 2), True),
    StructField("regiao", StringType(), True),
    StructField("tempo_clique_segundos", IntegerType(), True),
    StructField("interacoes_chat_suporte", IntegerType(), True),
    StructField("cupons_ativos_conta", IntegerType(), True),
    StructField("score_satisfacao_nps", IntegerType(), True),
    StructField("dias_desde_ultima_visita", IntegerType(), True)
])

tratado = (
    bronze_eventos
    .withColumn(
        "timestamp_registro",
        F.to_timestamp("timestamp_registro")
    )
    .withColumn(
        "payload",
        F.from_json(
            F.col("dados_origem_json"),
            schema_json
        )
    )
)

invalidos = tratado.filter(
    F.col("timestamp_registro").isNull()
    | F.col("payload.id_produto").isNull()
).limit(1).count()

if invalidos:
    raise ValueError(
        "Existem timestamps ou JSONs invalidos"
    )

df_silver = tratado.select(
    "id_evento",
    "id_usuario",
    "tipo_evento",
    "timestamp_registro",
    "payload.*"
).dropDuplicates(["id_evento"])

if df_silver.count() != n_eventos:
    raise RuntimeError("Perda de registros na Silver")

silver = salvar_sem_sobrescrever(
    df_silver, t_silver, n_eventos
)

print("CAMADA SILVER CONFERIDA")

## Etapa 5 — Gold: integração com a dimensão de produtos

A tabela Gold será criada pelo JOIN entre a Silver
e a dimensão de produtos existente na Bronze.

O `id_produto` permanece na Gold para rastreabilidade.

In [ ]:
df_gold = (
    silver.alias("s")
    .join(
        bronze_produtos.alias("p"),
        F.col("s.id_produto") == F.col("p.id_produto"),
        "inner"
    )
    .select(
        "s.*",
        F.col("p.categoria"),
        F.col("p.preco_reais")
    )
)

n_gold = df_gold.count()

if n_gold != n_eventos:
    raise RuntimeError(
        f"JOIN divergente: {n_gold} de {n_eventos}"
    )

gold = salvar_sem_sobrescrever(
    df_gold, t_gold, n_gold
)

print("CAMADA GOLD CONFERIDA:", gold.count())

## Etapa 6 — Auditoria final no Databricks

Confirma a existência e as contagens das quatro
tabelas Delta criadas ou reutilizadas.

In [ ]:
tabelas = [
    (t_eventos, 20000),
    (t_produtos, 15),
    (t_silver, 20000),
    (t_gold, 20000)
]

for tabela, esperado in tabelas:

    encontrado = spark.table(tabela).count()

    if encontrado != esperado:
        raise RuntimeError(
            f"Falha na validacao: {tabela}"
        )

    print("VALIDADA:", tabela, encontrado)

display(spark.table(t_gold).limit(20))

print("BRONZE, SILVER E GOLD: VALIDACAO CONCLUIDA")